[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/choROPeNt/FFTjax/blob/main/notebooks/structure-property_phi-sweep.ipynb)

# Structure-Property: Transverse Modulus vs. Fibre Volume Fraction

A minimal structure-property study: sweep the fibre volume fraction $\phi$ of a
random-fibre RVE (`generation.rve.make_random_composite_rve`,
[Catalanotti 2016](https://doi.org/10.1016/j.compstruct.2015.11.039)), solve for
the effective transverse Young's modulus $E(\phi)$ at each $\phi$ via
`learning.extractors.effective_modulus` (a mixed strain/stress boundary-condition
displacement solve -- same free-lateral-surface tensile test as
[Linear-Elastic Solve (mixed BC)](https://choROPeNt.github.io/FFTjax/documentation/examples/lin-elastic-mixed-bc)),
then fit a Gaussian-process surrogate $E(\phi)$ with `learning.surrogates.GPSurrogate`
([GPJax](https://docs.jaxgaussianprocesses.com/) underneath, same Matérn-kernel
exact-GP machinery `scripts/active_learning.py` uses) -- a smooth,
uncertainty-aware structure-property curve from a handful of FFT solves,
instead of a new solve every time an intermediate $\phi$ is needed.

The GP is refit **inside** the sweep loop, `N_UPDATES` times as data
accumulates, rather than once at the end -- at each update it's asked one
"simple answer" question (the modulus at a fixed, never-simulated query
$\phi$) so you can watch that cheap answer sharpen as more real FFT solves
feed into it, without ever running a new solve at the query point itself.

## Setup

In [ ]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("Running on Colab — installing FFTjax...")
    %pip install -q git+https://github.com/choROPeNt/FFTjax.git
else:
    import sys
    sys.path.insert(0, "../src")
    print("Running locally — using the local src/ checkout.")

import utils.precision  # side effect: configures JAX (X64 off on TPU, no GPU prealloc)
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

print("JAX backend:", jax.default_backend())
print("Devices:", jax.devices())
print("X64 enabled:", utils.precision.X64_ENABLED,
      "-> dtype:", jnp.zeros(1).dtype)

## Sweep $\phi$, updating the GP surrogate as solves arrive

Same random-fibre generator as
[pff-damage.ipynb](https://choROPeNt.github.io/FFTjax/documentation/examples/phase-field),
here with no interphase (binary matrix/fibre) since only the effective
elastic modulus is wanted. One fixed `seed` across the sweep isolates
$\phi$'s effect on $E$ from packing-realisation noise.

Boundary condition per $\phi$: $\varepsilon_{11}$ strain-controlled (small
uniaxial probe strain), $\varepsilon_{22}, \varepsilon_{33}$ stress-controlled
to zero (free lateral surfaces) -- a real transverse tensile test, exactly
`lin-elastic_mixed-BC.ipynb`'s `control`/`stress_goal` convention. The
resulting $E = \bar\sigma_{11} / \bar\varepsilon_{11}$ is the true engineering
transverse modulus, not the stiffer constrained coefficient a pure-strain BC
would give.

`N_UPDATES` controls how many times, evenly spaced through the sweep, the GP
below actually gets refit -- raise it to watch the surrogate update more
often (down to every single new solve), lower it to just a couple of
checkpoints. `PHI_QUERY` is a $\phi$ value that is **never** simulated here
-- at every update, the GP is asked for its mean/uncertainty at that one
point, which is the "simple answer" this notebook is actually demonstrating:
a cheap prediction, sharpening as real data comes in, standing in for a solve
that was never run.

In [ ]:
from generation.rve import make_random_composite_rve
from materialmodels.elastic.isotropic import LinearElasticIsotropic
from learning.extractors import effective_modulus
from learning.surrogates import GPSurrogate

r_fiber   = 0.005   # mm
vox       = 0.001   # mm -- 5 voxels per fibre radius
size_in_r = 10       # domain side ~ 10*r_fiber (Catalanotti 2016 convention)
seed      = 42       # fixed packing realisation across the sweep

E_matrix, nu_matrix = 3500.0, 0.35    # epoxy matrix
E_fiber,  nu_fiber  = 70000.0, 0.20   # glass fibre
eps0 = 1.0e-3   # small uniaxial tensile strain probe (xx)

N_UPDATES = 3      # how many times the GP gets refit as the sweep progresses
PHI_QUERY = 0.35   # never simulated -- the GP's "simple answer" target

phis        = np.linspace(0.10, 0.60, 12)
phi_grid    = np.linspace(phis.min(), phis.max(), 200)
snapshot_at = sorted(set(np.linspace(3, len(phis), N_UPDATES).round().astype(int)))

phi_obs, E_obs, nu_obs = [], [], []
gp_snapshots = []   # per update: n_obs, mean/std over phi_grid, mean/std at PHI_QUERY

for k, phi in enumerate(phis, start=1):
    phase_np, n, L, phi_act, _ = make_random_composite_rve(
        phi=phi, r_fiber=r_fiber, dx=vox, size_in_r=size_in_r, nz=1, K=15, seed=seed,
    )
    phase = jnp.array(phase_np.reshape(-1))
    materials = [
        LinearElasticIsotropic(E=E_matrix, nu=nu_matrix, name="epoxy matrix"),
        LinearElasticIsotropic(E=E_fiber, nu=nu_fiber, name="glass fibre"),
    ]

    E_val, nu_val, converged = effective_modulus(
        phase, materials, n, L, component=(0, 0), eps0=eps0, toler_lin=1e-6, maxiter=200,
    )
    phi_obs.append(phi_act)
    E_obs.append(E_val)
    nu_obs.append(nu_val[1])

    print(f"[{k:2d}/{len(phis)}] phi_act={phi_act:.3f}  grid={n}  "
          f"E={E_val:8.1f} MPa  nu={nu_val[1]:.4f}  converged={converged}")

    if k in snapshot_at:
        X_query = np.concatenate([phi_grid, [PHI_QUERY]])
        surrogate = GPSurrogate(num_iters=300).fit(phi_obs, E_obs, verbose=False)
        mean_all, var_all = surrogate.predict(X_query)
        std_all = np.sqrt(var_all)
        snap = {
            "n_obs": k, "mean_grid": mean_all[:-1], "std_grid": std_all[:-1],
            "mean_q": float(mean_all[-1]), "std_q": float(std_all[-1]),
        }
        gp_snapshots.append(snap)
        print(f"          -> GP update ({k} obs): E(phi={PHI_QUERY}) = "
              f"{snap['mean_q']:.1f} +/- {2 * snap['std_q']:.1f} MPa (95%, never simulated)")

phi_obs, E_obs, nu_obs = np.array(phi_obs), np.array(E_obs), np.array(nu_obs)

## Visualize: the GP surrogate tightening, and its answer at $\phi=$ PHI_QUERY

Left: every GP snapshot's mean curve over the sweep range (darker = more
observations), the final snapshot's $\pm2\sigma$ band, and the actual FFT
solves it was fit on. Right: the GP's answer at the never-simulated
`PHI_QUERY`, plotted against how many real solves it had seen at that
point -- this is the "simple answer" getting cheaper and more confident
with no new FFT solve at $\phi=$ PHI_QUERY itself.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

cmap = plt.cm.Blues
n_snap = len(gp_snapshots)
for i, snap in enumerate(gp_snapshots):
    color = cmap(0.35 + 0.55 * (i + 1) / n_snap)
    axes[0].plot(phi_grid, snap["mean_grid"], "-", color=color,
                 label=f"GP after {snap['n_obs']} obs")

final = gp_snapshots[-1]
axes[0].fill_between(phi_grid, final["mean_grid"] - 2 * final["std_grid"],
                      final["mean_grid"] + 2 * final["std_grid"],
                      color=cmap(0.9), alpha=0.15, label=r"final GP $\pm2\sigma$")
axes[0].plot(phi_obs, E_obs, "o", color="black", markersize=5, label="FFT solves")
axes[0].axvline(PHI_QUERY, color="gray", ls="--", lw=1, label=fr"query $\phi={PHI_QUERY}$")
axes[0].set_xlabel(r"$\phi$ (fibre volume fraction)")
axes[0].set_ylabel(r"$E_{11}$ [MPa]")
axes[0].set_title("GP surrogate tightening with more solves")
axes[0].legend(fontsize=8)
axes[0].grid(True, linewidth=0.5, alpha=0.6)

n_obs_hist  = [s["n_obs"] for s in gp_snapshots]
mean_q_hist = [s["mean_q"] for s in gp_snapshots]
std_q_hist  = [s["std_q"] for s in gp_snapshots]
axes[1].errorbar(n_obs_hist, mean_q_hist, yerr=2 * np.array(std_q_hist),
                  fmt="o-", color="C0", capsize=4)
axes[1].set_xlabel("number of FFT solves used")
axes[1].set_ylabel(fr"GP prediction $E(\phi={PHI_QUERY})$ [MPa]")
axes[1].set_title("Cheap, uncertainty-aware answer -- no new solve needed")
axes[1].grid(True, linewidth=0.5, alpha=0.6)

fig.tight_layout()
plt.show()

## Next steps

- Lower `N_UPDATES` to see fewer, coarser refits, or raise it toward
  `len(phis)` to update the GP after every single new solve.
- Use the GP's posterior variance to pick the *next* $\phi$ to actually
  simulate (uncertainty sampling) instead of an evenly-spaced sweep --
  exactly `scripts/active_learning.py`'s active-learning loop, applied here
  to $\phi$ instead of a VAE latent code.
- Add a second sweep dimension (e.g. `r_fiber`, or the matrix/fibre modulus
  ratio) -- `gpx.kernels.Matern52` handles a multi-dimensional `X` unchanged,
  same as `scripts/active_learning.py`'s 4-D latent-space GP.
- Swap the isotropic glass fibre for `TransverseIsotropic` carbon fibre
  (as in `pff-damage.ipynb`) and compare the longitudinal vs. transverse
  modulus sweep.